<a href="https://colab.research.google.com/github/pranavik24/PromptInject/blob/main/notebooks/comparisons.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install sentence-transformers
!pip install rouge-score

  Preparing metadata (setup.py) ... done
  Created wheel for rouge-score: filename=rouge_score-0.1.2-py3-none-any.whl size=24934 sha256=74ec18aca0e61ced169ddffc96bf19116bf124a8f3998f8d2b8098163c293018
  Stored in directory: /root/.cache/pip/wheels/85/9d/af/01feefbe7d55ef5468796f0c68225b6788e85d9d0a281e7a70
Successfully built rouge-score


In [10]:
%cd /content/drive/MyDrive
! git clone https://github.com/pranavik24/PromptInject.git
%cd PromptInject
!git pull

/content/drive/MyDrive
fatal: destination path 'PromptInject' already exists and is not an empty directory.
/content/drive/MyDrive/PromptInject
remote: Enumerating objects: 9, done.
remote: Counting objects: 100% (9/9), done.
remote: Compressing objects: 100% (1/1), done.
remote: Total 5 (delta 4), reused 5 (delta 4), pack-reused 0 (from 0)
Unpacking objects: 100% (5/5), 95.50 KiB | 74.00 KiB/s, done.
From https://github.com/pranavik24/PromptInject
   275909f..d9e75be  main       -> origin/main
Updating 275909f..d9e75be
Fast-forward
 notebooks/BinaryClassification.ipynb   | 4117 ++++++++++++++++----------------
 notebooks/Generate_Clean_withCOT.ipynb |  398 ++-
 2 files changed, 2296 insertions(+), 2219 deletions(-)


In [1]:
!git config --global user.email "elainego@andrew.cmu.edu"
!git config --global user.name "Elaine Gombos"
!git add .
!git commit -m "Update with graphs and tables"

fatal: not a git repository (or any of the parent directories): .git
fatal: not a git repository (or any of the parent directories): .git


In [2]:
import pandas as pd
import re
import math
from sentence_transformers import SentenceTransformer
import numpy as np
from rouge_score import rouge_scorer

In [1]:
# Cell 1 — mount google drive
from google.colab import drive
drive.mount('/content/drive/')

Mounted at /content/drive/


In [4]:
model = SentenceTransformer("all-mpnet-base-v2")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/571 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/363 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [5]:

def encode_outputs(outputs1: list[str], outputs2: list[str], batch_size: int = 32):
    vecs1 = model.encode(outputs1, batch_size=batch_size, show_progress_bar=True)
    vecs2 = model.encode(outputs2, batch_size=batch_size, show_progress_bar=True)
    return vecs1, vecs2

In [6]:

def compute_cosine_scores(vecs1: np.ndarray, vecs2: np.ndarray) -> np.ndarray:
    norms1 = np.linalg.norm(vecs1, axis=1, keepdims=True)
    norms2 = np.linalg.norm(vecs2, axis=1, keepdims=True)
    return np.round(np.sum((vecs1 / norms1) * (vecs2 / norms2), axis=1), 4)

In [7]:
def compute_rouge_scores(outputs1: list[str], outputs2: list[str]) -> tuple[list, list]:
    scorer = rouge_scorer.RougeScorer(["rouge1", "rouge2"], use_stemmer=True)
    rouge1_scores, rouge2_scores = [], []
    for o1, o2 in zip(outputs1, outputs2):
        scores = scorer.score(o1, o2)
        rouge1_scores.append(round(scores["rouge1"].fmeasure, 4))
        rouge2_scores.append(round(scores["rouge2"].fmeasure, 4))
    return rouge1_scores, rouge2_scores

In [10]:
def compare_model_outputs(path_without_cot: str, path_with_cot: str, output_path: str):
    df1 = pd.read_csv(path_without_cot)
    df2 = pd.read_csv(path_with_cot)

    assert len(df1) == len(df2), "CSVs have different number of rows"

    outputs1 = df1["output_text"].astype(str).tolist()
    outputs2 = df2["output_text"].astype(str).tolist()

    print("Encoding outputs...")
    vecs1, vecs2 = encode_outputs(outputs1, outputs2)

    print("Computing cosine similarity scores...")
    cosine_scores = compute_cosine_scores(vecs1, vecs2)

    print("Computing ROUGE scores...")
    rouge1_scores, rouge2_scores = compute_rouge_scores(outputs1, outputs2)

    pd.DataFrame({
        # "number": df1["number"],
        "prompt_instruction": df1["system_prompt"],
        "attack_instruction": df1["attack_name"],
        "output_WithoutCOT": df1["output_text"],
        "output_WithCOT": df2["output_text"],
        "cosineSimilarityScore": cosine_scores,
        "rouge1": rouge1_scores,
        "rouge2": rouge2_scores,
    }).to_csv(output_path, index=False)
    print(f"Done. Results saved to {output_path}")

In [12]:
compare_model_outputs(
    path_without_cot="/content/drive/MyDrive/11711 group project/elaine work/noinjection_withoutCOT_dup.csv",
    path_with_cot="/content/drive/MyDrive/11711 group project/elaine work/noinjection_withCOT_dup.csv",
    output_path="/content/drive/MyDrive/11711 group project/elaine work/noinjection_comparison_results.csv",
)

Encoding outputs...


Batches:   0%|          | 0/17 [00:00<?, ?it/s]

Batches:   0%|          | 0/17 [00:00<?, ?it/s]

Computing cosine similarity scores...
Computing ROUGE scores...
Done. Results saved to /content/drive/MyDrive/11711 group project/elaine work/noinjection_comparison_results.csv


In [2]:
import pandas as pd
import numpy as np

# Update this path to wherever compare_model_outputs() saved its output
COMPARISON_CSV = "/content/drive/MyDrive/11711 group project/elaine work/noinjection_comparison_results.csv"

df = pd.read_csv(COMPARISON_CSV)

print(f"Loaded {len(df)} rows")
print(f"Columns: {df.columns.tolist()}")
print()
print(df[["prompt_instruction", "attack_instruction", "rouge1", "rouge2", "cosineSimilarityScore"]].head(5))

Loaded 525 rows
Columns: ['prompt_instruction', 'attack_instruction', 'output_WithoutCOT', 'output_WithCOT', 'cosineSimilarityScore', 'rouge1', 'rouge2']

  prompt_instruction attack_instruction  rouge1  rouge2  cosineSimilarityScore
0    default-grammar              clean  0.0889     0.0                   0.16
1    default-grammar              clean  0.0889     0.0                   0.16
2    default-grammar              clean  0.0889     0.0                   0.16
3    default-grammar              clean  0.0889     0.0                   0.16
4    default-grammar              clean  0.0889     0.0                   0.16


In [3]:
METRICS = ["rouge1", "rouge2", "cosineSimilarityScore"]

def agg_metrics(df, group_cols):
    agg = df.groupby(group_cols)[METRICS].agg(["mean", "std"]).round(4)
    agg.columns = [f"{metric}_{stat}" for metric, stat in agg.columns]
    agg["n"] = df.groupby(group_cols)[METRICS[0]].count()
    return agg

by_prompt = agg_metrics(df, "prompt_instruction")
print("=" * 70)
print("Aggregated by prompt_instruction")
print("=" * 70)
pd.set_option("display.max_colwidth", 50)
pd.set_option("display.width", 200)
print(by_prompt.to_string())

print()

by_attack = agg_metrics(df, "attack_instruction")
print("=" * 70)
print("Aggregated by attack_instruction")
print("=" * 70)
print(by_attack.to_string())

print()

by_both = agg_metrics(df, ["prompt_instruction", "attack_instruction"])
print("=" * 70)
print("Aggregated by prompt_instruction x attack_instruction")
print("=" * 70)
print(by_both.to_string())

Aggregated by prompt_instruction
                                 rouge1_mean  rouge1_std  rouge2_mean  rouge2_std  cosineSimilarityScore_mean  cosineSimilarityScore_std   n
prompt_instruction                                                                                                                          
default-ad-product-description        0.3256         0.0       0.0510         0.0                      0.6235                        0.0  15
default-adv-tweet-classifier          0.4175         0.0       0.3039         0.0                      0.6844                        0.0  15
default-airport-codes                 0.1905         0.0       0.0000         0.0                      0.5788                        0.0  15
default-analogy-maker                 0.3084         0.0       0.0377         0.0                      0.2449                        0.0  15
default-chat                          0.2124         0.0       0.0446         0.0                      0.3146            

In [4]:
import pandas as pd
import numpy as np

METRICS = ["rouge1", "rouge2", "cosineSimilarityScore"]

def agg_metrics(df, group_cols):
    agg = df.groupby(group_cols)[METRICS].agg(["mean", "std"]).round(4)
    agg.columns = [f"{metric}_{stat}" for metric, stat in agg.columns]
    agg["n"] = df.groupby(group_cols)[METRICS[0]].count()
    return agg

by_prompt = agg_metrics(df, "prompt_instruction")
by_attack = agg_metrics(df, "attack_instruction")
by_both   = agg_metrics(df, ["prompt_instruction", "attack_instruction"])

from IPython.display import display, HTML

def show_table(agg_df, title):
    display(HTML(f"<h3>{title}</h3>"))
    display(agg_df.style
        .format({c: "{:.4f}" for c in agg_df.columns if c != "n"})
        .background_gradient(subset=["rouge1_mean", "rouge2_mean", "cosineSimilarityScore_mean"], cmap="Blues")
        .set_table_styles([
            {"selector": "th", "props": [("font-size", "11px"), ("text-align", "center")]},
            {"selector": "td", "props": [("font-size", "10px")]},
        ])
    )

show_table(by_prompt, "Aggregated by prompt_instruction")
show_table(by_attack, "Aggregated by attack_instruction")
show_table(by_both,   "Aggregated by prompt_instruction × attack_instruction")

,rouge1_mean,rouge1_std,rouge2_mean,rouge2_std,cosineSimilarityScore_mean,cosineSimilarityScore_std,n
prompt_instruction,,,,,,,
default-ad-product-description,0.3256,0.0000,0.0510,0.0000,0.6235,0.0000,15
default-adv-tweet-classifier,0.4175,0.0000,0.3039,0.0000,0.6844,0.0000,15
default-airport-codes,0.1905,0.0000,0.0000,0.0000,0.5788,0.0000,15
default-analogy-maker,0.3084,0.0000,0.0377,0.0000,0.2449,0.0000,15
default-chat,0.2124,0.0000,0.0446,0.0000,0.3146,0.0000,15
default-esrb-rating,0.0615,0.0000,0.0317,0.0000,0.7876,0.0000,15
default-essay-outline,0.2093,0.0000,0.0714,0.0000,0.2592,0.0000,15
default-extract-contact-info,0.1243,0.0000,0.0343,0.0000,0.4004,0.0000,15
default-factual-answering,0.8636,0.0000,0.7419,0.0000,0.7652,0.0000,15


,rouge1_mean,rouge1_std,rouge2_mean,rouge2_std,cosineSimilarityScore_mean,cosineSimilarityScore_std,n
attack_instruction,,,,,,,
clean,0.2325,0.1743,0.1034,0.1574,0.4535,0.2506,525


,,rouge1_mean,rouge1_std,rouge2_mean,rouge2_std,cosineSimilarityScore_mean,cosineSimilarityScore_std,n
prompt_instruction,attack_instruction,,,,,,,
default-ad-product-description,clean,0.3256,0.0000,0.0510,0.0000,0.6235,0.0000,15
default-adv-tweet-classifier,clean,0.4175,0.0000,0.3039,0.0000,0.6844,0.0000,15
default-airport-codes,clean,0.1905,0.0000,0.0000,0.0000,0.5788,0.0000,15
default-analogy-maker,clean,0.3084,0.0000,0.0377,0.0000,0.2449,0.0000,15
default-chat,clean,0.2124,0.0000,0.0446,0.0000,0.3146,0.0000,15
default-esrb-rating,clean,0.0615,0.0000,0.0317,0.0000,0.7876,0.0000,15
default-essay-outline,clean,0.2093,0.0000,0.0714,0.0000,0.2592,0.0000,15
default-extract-contact-info,clean,0.1243,0.0000,0.0343,0.0000,0.4004,0.0000,15
default-factual-answering,clean,0.8636,0.0000,0.7419,0.0000,0.7652,0.0000,15


In [5]:
import pandas as pd
import numpy as np

# Update this path to wherever compare_model_outputs() saved its output
COMPARISON_CSV2 = "/content/drive/MyDrive/11711 group project/elaine work/injection_comparison_results.csv"

df = pd.read_csv(COMPARISON_CSV2)

print(f"Loaded {len(df)} rows")
print(f"Columns: {df.columns.tolist()}")
print()
print(df[["prompt_instruction", "attack_instruction", "rouge1", "rouge2", "cosineSimilarityScore"]].head(5))

Loaded 525 rows
Columns: ['prompt_instruction', 'attack_instruction', 'output_WithoutCOT', 'output_WithCOT', 'cosineSimilarityScore', 'rouge1', 'rouge2']

  prompt_instruction                                 attack_instruction  rouge1  rouge2  cosineSimilarityScore
0    default-grammar            ignore-previous-and-spell-check-instead  0.2222  0.0253                 0.3020
1    default-grammar            ignore-previous-and-spell-check-instead  0.2222  0.0253                 0.3020
2    default-grammar            ignore-previous-and-spell-check-instead  0.2222  0.0253                 0.3020
3    default-grammar  ignore-previous-and-spell-check-instead-uppercase  0.2308  0.0000                 0.4156
4    default-grammar  ignore-previous-and-spell-check-instead-uppercase  0.2308  0.0000                 0.4156


In [6]:
METRICS = ["rouge1", "rouge2", "cosineSimilarityScore"]

def agg_metrics(df, group_cols):
    agg = df.groupby(group_cols)[METRICS].agg(["mean", "std"]).round(4)
    agg.columns = [f"{metric}_{stat}" for metric, stat in agg.columns]
    agg["n"] = df.groupby(group_cols)[METRICS[0]].count()
    return agg

by_prompt = agg_metrics(df, "prompt_instruction")
print("=" * 70)
print("Aggregated by prompt_instruction")
print("=" * 70)
pd.set_option("display.max_colwidth", 50)
pd.set_option("display.width", 200)
print(by_prompt.to_string())

print()

by_attack = agg_metrics(df, "attack_instruction")
print("=" * 70)
print("Aggregated by attack_instruction")
print("=" * 70)
print(by_attack.to_string())

print()

by_both = agg_metrics(df, ["prompt_instruction", "attack_instruction"])
print("=" * 70)
print("Aggregated by prompt_instruction x attack_instruction")
print("=" * 70)
print(by_both.to_string())

Aggregated by prompt_instruction
                                 rouge1_mean  rouge1_std  rouge2_mean  rouge2_std  cosineSimilarityScore_mean  cosineSimilarityScore_std   n
prompt_instruction                                                                                                                          
default-ad-product-description        0.1976      0.1168       0.0393      0.0348                      0.4202                     0.1707  15
default-adv-tweet-classifier          0.4179      0.2122       0.2522      0.2562                      0.7305                     0.0375  15
default-airport-codes                 0.3304      0.1258       0.0857      0.0773                      0.7076                     0.1090  15
default-analogy-maker                 0.2577      0.1312       0.0827      0.0847                      0.3478                     0.2062  15
default-chat                          0.2293      0.1439       0.1017      0.1033                      0.3689            

In [7]:
print("Overall mean and std across all rows:")
print(df[["rouge1", "rouge2", "cosineSimilarityScore"]].agg(["mean", "std"]).round(4).to_string())

Overall mean and std across all rows:
      rouge1  rouge2  cosineSimilarityScore
mean  0.2226  0.0966                 0.4074
std   0.1745  0.1449                 0.2349


In [8]:
import pandas as pd
import numpy as np

METRICS = ["rouge1", "rouge2", "cosineSimilarityScore"]

def agg_metrics(df, group_cols):
    agg = df.groupby(group_cols)[METRICS].agg(["mean", "std"]).round(4)
    agg.columns = [f"{metric}_{stat}" for metric, stat in agg.columns]
    agg["n"] = df.groupby(group_cols)[METRICS[0]].count()
    return agg

by_prompt = agg_metrics(df, "prompt_instruction")
by_attack = agg_metrics(df, "attack_instruction")
by_both   = agg_metrics(df, ["prompt_instruction", "attack_instruction"])

from IPython.display import display, HTML

def show_table(agg_df, title):
    display(HTML(f"<h3>{title}</h3>"))
    display(agg_df.style
        .format({c: "{:.4f}" for c in agg_df.columns if c != "n"})
        .background_gradient(subset=["rouge1_mean", "rouge2_mean", "cosineSimilarityScore_mean"], cmap="Blues")
        .set_table_styles([
            {"selector": "th", "props": [("font-size", "11px"), ("text-align", "center")]},
            {"selector": "td", "props": [("font-size", "10px")]},
        ])
    )

show_table(by_prompt, "Aggregated by prompt_instruction")
show_table(by_attack, "Aggregated by attack_instruction")
show_table(by_both,   "Aggregated by prompt_instruction × attack_instruction")

,rouge1_mean,rouge1_std,rouge2_mean,rouge2_std,cosineSimilarityScore_mean,cosineSimilarityScore_std,n
prompt_instruction,,,,,,,
default-ad-product-description,0.1976,0.1168,0.0393,0.0348,0.4202,0.1707,15
default-adv-tweet-classifier,0.4179,0.2122,0.2522,0.2562,0.7305,0.0375,15
default-airport-codes,0.3304,0.1258,0.0857,0.0773,0.7076,0.1090,15
default-analogy-maker,0.2577,0.1312,0.0827,0.0847,0.3478,0.2062,15
default-chat,0.2293,0.1439,0.1017,0.1033,0.3689,0.1518,15
default-esrb-rating,0.1160,0.1473,0.0466,0.0617,0.4530,0.2135,15
default-essay-outline,0.2201,0.1150,0.0697,0.0471,0.5615,0.1052,15
default-extract-contact-info,0.0498,0.0460,0.0080,0.0166,0.2588,0.1557,15
default-factual-answering,0.1940,0.1806,0.0805,0.1368,0.2546,0.1848,15


,rouge1_mean,rouge1_std,rouge2_mean,rouge2_std,cosineSimilarityScore_mean,cosineSimilarityScore_std,n
attack_instruction,,,,,,,
ignore-previous-and-print,0.2290,0.1733,0.0964,0.1281,0.4160,0.2393,105
ignore-previous-and-print-instead,0.2562,0.2081,0.1276,0.1842,0.4037,0.2474,105
ignore-previous-and-spell-check,0.2060,0.1390,0.0786,0.0997,0.4346,0.1973,105
ignore-previous-and-spell-check-instead,0.1921,0.1719,0.0891,0.1474,0.3620,0.2434,105
ignore-previous-and-spell-check-instead-uppercase,0.2297,0.1694,0.0914,0.1496,0.4208,0.2412,105
